##Water is ... Fascinating!!

The two extreme scenarios of molecular interaction are solids, having molecules with strong interactions, which are aligned in a rigid crystal, and gases, having molecules with no to minimal interactions, with essentially no regular arrangement in their structure. The amount of interaction (and thus structure) of molecules in a liquid falls in the middle. This is one of the things that makes liquids so interesting - ***and the main reason almost all chemistry occurs in a solvent!***

The structure in a liquid can be characterized by a radial distribution function (RDF), g(r). An RDF is a measure of the radial frequency of particles around some point in space. Typically, we use RDFs to measure the likelihood of finding an atom type (e.g. oxygen) around another atom type (e.g. hydrogen). For a monatomic gas, g(r) is flat, because any atom is a random distance from all other atoms. There is no preferred distance; all are equally likely. For a crystalline solid, g(r) has regular, sharp peaks, because atoms are regularly spaced from each other - you will be familiar with this from X-ray diffraction patterns. In a solid, only some distances are likely; others are essentially forbidden.

Liquids, on the other hand, are structured locally but random at long range. This means a liquid g(r) resembles that of a solid at short-range, but that of a gas at long-range. Most liquids typically exhibit a few distinct short range peaks, these correspond to **solvation shells**. The position of these peaks indicate the length scale of each solvation shell, and the height of that peak indicates how pronounced the corresponding shell is in structure, relative to the surrounding bulk liquid environment. Eventually, as the nonlocal density becomes identical to the bulk density, g(r) flattens and tends to the bulk value of 1.

For water, the amount of local structure can be characterized via the O-O, O-H and H-H g(r) functions (i.e. "oxygens around oxygens", "hydrogens around oxygens" and "hyrogens around hydrogens"), which are shown in the Figure below. This figure is produced via neutron diffraction measurements, however MD is also capable of producing RDFs. These RDFs show that:

1. The hydrogen bonding structure in bulk water (i.e. the O-H g(r))  extends well beyond the first solvation shell (there are two hydrogen bond interactions at ~2 and 3.5 $\mathring A$, respectively).
1. The interaction length between neighbouring water molecules (via the O-O g(r)). The figure below shows that water arranges itself into ~3 distinct solvation shells at ~2.5, 4.5 and 7 $\mathring A$, respectively.

So, water is **really** structured!! Now that we understand this, we can begin to understand the physical and chemical properties of water, and why some of them change in the presence of solutes and interfaces (as we will see in the next workshop).

<img src="https://www.pnas.org/cms/10.1073/pnas.1418982112/asset/32a96abe-ab76-4c69-9446-af3cb00a8a4b/assets/graphic/pnas.1418982112fig03.jpeg" align="left" width="800" height="300"/>

In this exercise, we will use MD simulations to analyse the RDFs of the principal interactions in liquid water more closely.

Use the MD calculator below to consider the following questions.

1. Does the TIP3P water model reproduce the principal O-O, O-H and H-H interactions in liquid water?
1. Does the TIP4P-Ew water model do any better than the TIP3P modeL? If so, why? (If not, why not?) (_HINT - consider the assumptions and terms that define these two water models_)
1. At what length scale does the agreement between your predicted structure and experimental structure break down? Can this be fixed by running larger and/or longer simulations?

In [ ]:
#@title Setup the Environment
!python -V
!pip install -q openmm MDAnalysis

# install compiler
!apt-get -qq update
!apt-get -qq install -y gfortran

# remove existing directory if it exists
import os
import shutil
if os.path.exists("packmol-master"):
    print("Removing existing packmol-master directory...")
    shutil.rmtree("packmol-master")

# download latest Packmol source
!wget -q https://github.com/m3g/packmol/archive/refs/heads/master.zip
!unzip -q master.zip

# compile
%cd packmol-master
!make
%cd ..

# add to PATH

os.environ["PATH"] = "/content/packmol-master:" + os.environ["PATH"]
!which packmol

from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
import re
import time
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#@title Choose Simulation Parameters 

import ipywidgets as widgets
from IPython.display import display

#SIMULATION TEMPERATURE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=298.0,  # default value
    description='Simulation Temperature (K):',
    step=1.0,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simTemperature
    simTemperature = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simTemperature = temp_text.value*kelvin

#BOXSIZE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=26.0,  # default value
    description='Box Size (Angstrom):',
    step=1.0,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global boxsize
    boxsize = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
boxsize = temp_text.value

#TIMESTEP
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=1.0,  # default value
    description='Time step (fs):',
    step=0.5,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simTimestep
    simTimestep = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simTimestep = temp_text.value*femtoseconds

#PRESSURE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=1.0,  # default value
    description='Simulation pressure (atm):',
    step=0.10,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simPressure
    simPressure = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simPressure = temp_text.value*atmospheres

#STEPS
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=100000,  # default value
    description='# MD steps:',
    step=100,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simNumSteps
    simNumSteps = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simNumSteps = temp_text.value

#Forcefield selection
ffield_dropdown = widgets.Dropdown(
    options=['TIP3P', 'TIP4P-Ew'],
    value='TIP3P', # Default value
    description='Forcefield:',
    disabled=False,
)
ffield_dropdown.layout.width = 'auto'
ffield_dropdown.style.description_width = 'initial'

def update_ffield(change):
    global ffield
    selected_value = change.new
    if selected_value == 'TIP3P':
        ffield = 'tip3p'
    elif selected_value == 'TIP4P-Ew':
        ffield = 'tip4pew'

ffield_dropdown.observe(update_ffield, 'value')
display(ffield_dropdown)

# Set initial ffield based on the default value of the dropdown
initial_ffield_value = ffield_dropdown.value
if initial_ffield_value == 'TIP3P':
    ffield = 'tip3p'
elif initial_ffield_value == 'TIP4P-Ew':
    ffield = 'tip4pew'


In [ ]:
#@title Parameter Summary

density=0.99705
water=int(boxsize**3*density*1e6*(1e-10)**3/(18.01)*6.02214e23)
print("Box size of",boxsize,"angstroms at density,",density,"g/cm3 requires ",water," water molecules")
print(f"Simulation size: {water} waters")
print(f"Box size: {boxsize} Ang.")
print(f"Simulation temperature: {simTemperature}")
print(f"Simulation pressure: {simPressure}")
print(f"Simulation time step: {simTimestep}")
print(f"# MD steps: {simNumSteps} ; Total simulation length: {simNumSteps * simTimestep} = {simNumSteps*simTimestep / 1000 / 1000 / femtoseconds * nanoseconds}.")
print(f"Water model: {ffield}")
pdbFile = 'system.pdb'

#clean up any possible previous runs
for f in pdbFile, 'system_tip4pew.pdb', 'data.dcd', 'data.csv', 'water.pdb', 'packmol.in', 'packmol.out':
  if os.path.exists(f):
    os.remove(f)



In [ ]:
#@title Setup the Environment 🔨🔨🔨

with open("packmol.in", "w") as f:
    f.write("tolerance 2.0\n")
    f.write("filetype pdb\n")
    f.write("output system.pdb\n")
    f.write("\n")
    f.write("structure water.pdb\n")
    f.write("  number {0}\n".format(int(water)))
    f.write("  inside cube 2. 2. 2. {0}\n".format(boxsize))
    f.write("end structure\n")
    f.write("\n")
    f.write("pbc {0} {0} {0}\n".format(boxsize))

text = '''ATOM      1  O   HOH A   1       4.125  13.679  13.761  1.00  0.00           O
ATOM      2  H1  HOH A   1       4.025  14.428  14.348  1.00  0.00           H
ATOM      3  H2  HOH A   1       4.670  13.062  14.249  1.00  0.00           H'''

with open('water.pdb', 'w') as f:
    f.write(text)

!packmol < packmol.in > packmol.out

In [ ]:
#@title Setup & Run MD Simulation 🔨🔨🔨


pdb = PDBFile('system.pdb')
forcefield = ForceField(f"{ffield}.xml")


if ffield == 'tip4pew':
  modeller = Modeller(pdb.topology, pdb.positions)
  modeller.addExtraParticles(forcefield)
  PDBFile.writeFile(modeller.topology, modeller.positions, open('system_tip4pew.pdb', 'w'))
  pdb = PDBFile('system_tip4pew.pdb','M')

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=8*angstrom,
    constraints=HBonds,
    rigidWater=True)

system.addForce(MonteCarloBarostat(simPressure, simTemperature, 1))

integrator = LangevinMiddleIntegrator(simTemperature, 1/picosecond, simTimestep)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)

simulation.reporters.append(DCDReporter('data.dcd', 10))
simulation.reporters.append(StateDataReporter(
    'data.csv',
    10,
    step=True,
    temperature=True,
    potentialEnergy=True,
    kineticEnergy=True,
    totalEnergy=True,
    volume=True,
    density=True
))

print("Running Energy Minimisation:")
t0 = time.time()
simulation.minimizeEnergy()
t1 = time.time()
minTime = t1-t0
print(f"{ffield} minimisation took {minTime} seconds")

print('Running NPT Simulation')
t0 = time.time()
simulation.step(simNumSteps)
t1 = time.time()
simTime = t1-t0
print(f"{ffield} simulation took {simTime} seconds for {simNumSteps} timesteps")


In [ ]:
#@title Are we Equilibrated Yet?🧪🧪🧪


data = np.genfromtxt('data.csv', skip_header=1,delimiter=',')
steps = data[:,0]
temperature = data[:,4]
density = data[:,6]

# Create a 2x1 subplot
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Plot Temperature vs. Timestep
axes[0].plot(steps, temperature, color='blue')
axes[0].set_ylabel('Temperature (K)')
axes[0].set_title('Temperature and Density vs. Simulation Timestep')
axes[0].grid(True, linestyle='--', alpha=0.7)

# Plot Density vs. Timestep
axes[1].plot(steps, density, color='red')
axes[1].set_xlabel('Simulation Timestep')
axes[1].set_ylabel('Density (g/cm^3)')
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

average_density=np.mean(density[-20000:])
stdev_density=np.std(density[-20000:])
print("Density is",average_density,"-/-",stdev_density," g/cm3")
average_temperature=np.mean(temperature[-20000:])
stdev_temperature=np.std(temperature[-20000:])
print("Temperature is",average_temperature,"-/-",stdev_temperature," K")

In [ ]:
#@title Calculate Radial Distribution Functions 💻
import MDAnalysis as mda
import MDAnalysis.analysis.rdf as rdf
from tqdm.notebook import tqdm

# Determine the correct PDB topology file based on the forcefield selected in the parameters cell
pdb_topology_file = 'system.pdb'
if 'ffield' in globals() and ffield == 'tip4pew':
  pdb_topology_file = 'system_tip4pew.pdb'

u = mda.Universe(pdb_topology_file,'data.dcd')
O = u.select_atoms('name O')
H = u.select_atoms('name H1 or name H2')

# Initialize RDF objects
OO_rdf = rdf.InterRDF(O,O,nbins=100,range=(1.0,10))
OH_rdf = rdf.InterRDF(O,H,nbins=100,range=(1.0,10))
HH_rdf = rdf.InterRDF(H,H,nbins=100,range=(1.0,10))

# Run the RDF calculations. The 'frames' argument accepts an iterable,
# so we can wrap u.trajectory with tqdm for a progress bar.
OO_rdf.run(progressbar_kwargs={'desc': "Calculating O-O RDF"})
OH_rdf.run(progressbar_kwargs={'desc': "Calculating O-H RDF"})
HH_rdf.run(progressbar_kwargs={'desc': "Calculating H-H RDF"})


In [ ]:
#@title Plot the Radial Distribution Functions 💻

import matplotlib.pyplot as plt

plt.plot(OO_rdf.bins,OO_rdf.rdf,label="O-O")
plt.plot(OH_rdf.bins,OH_rdf.rdf,label="O-H")
plt.plot(HH_rdf.bins,HH_rdf.rdf,label="H-H")
plt.xlabel("r ($\mathrm{\AA}$)")
plt.ylabel("g(r)")
plt.legend()
plt.ylim([0.0, 5])
plt.xlim([0.0,10])